In [ ]:
!ls

gru4rec_config.json	   GRU4Rec_PyTorch_Official___
GRU4Rec_PyTorch_Official   gru_pt_10_32_ce_L100_model_.pt
GRU4Rec_PyTorch_Official_


In [ ]:
import os
import shutil
import sys
import pandas as pd
import datetime
from google.cloud import bigquery
import subprocess
import glob
print(os.getcwd())

sys.path.insert(0,'./GRU4Rec_PyTorch_Official/')
#WEB_RECS_DERIVED.gru_training_data

project = "prod-analytics-recommend-c1jeg"
dataset = "RED_RECS"
table_prefix = "TRAINING_SET_TEST"
table_suffix = ''
#os.environ["GCLOUD_PROJECT"] = 'expanded-nebula-754'
os.environ["GCLOUD_PROJECT"] = 'prod-analytics-recommend-c1jeg'

import os.path
orig_cwd = os.getcwd()
import numpy as np
import json
import time
from collections import OrderedDict
import importlib
GRU4Rec = importlib.import_module('gru4rec_pytorch').GRU4Rec
import evaluation
import importlib.util
import joblib
import gc
os.chdir(orig_cwd)

#device = 'cuda:0'
sample_store_size= 10000000

import os
import shutil
import sys
from gru4rec_pytorch import SessionDataIterator
import torch
os.chdir(orig_cwd)
#os.environ["GCLOUD_PROJECT"] = 'expanded-nebula-754'
os.environ["GCLOUD_PROJECT"] = 'prod-analytics-recommend-c1jeg'

import gc
print(torch.cuda.is_available())

/content
True


In [ ]:
# To fix the "PERMISSION_DENIED" error, you need to specify a 'pipeline_root'
# This prevents the SDK from trying to create a default bucket which you don't have permissions for.

from google.cloud import aiplatform

# TODO: Replace with a GCS bucket you have write access to
# You can list buckets using: !gcloud storage buckets list --project prod-analytics-recommend-c1jeg
BUCKET_NAME = "cdow"
PIPELINE_ROOT = f"gs://{BUCKET_NAME}/pipeline_root"

# Example of how to submit the job with the fix:
# job = aiplatform.PipelineJob(
#     display_name="gru4rec-pipeline",
#     template_path="gru4rec_pipeline.json", # Or your pipeline file
#     pipeline_root=PIPELINE_ROOT,           # <--- Add this argument
#     project="prod-analytics-recommend-c1jeg",
#     location="us-central1",                # Ensure location matches
# )

# job.submit()

/usr/local/lib/python3.10/dist-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.cloud.resourcemanager_v3 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.resourcemanager_v3 past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
def get_bq_training_table():#table_suffix, gcs_location, data_models_path):
  query = """
          SELECT * FROM `prod-analytics-recommend-c1jeg.RED_RECS.GRU_TRAINING_DATA`
        """
#expanded-nebula-754.sandbox_crdow.GRU_Pytorch_PDP_1SELECT * FROM `prod-analytics-recommend-c1jeg.RED_RECS.GRU_TRAINING_DATA`
#query = """prod-analytics-recommend-c1jeg.RED_RECS.GRU_TRAINING_DATA
#SELECT * FROM `prod-analytics-recommend-c1jeg.RED_RECS.GRU_TRAINING_DATA` prod-analytics-recommend-c1jeg.RED_RECS.GRU_TRAINING_DATA
#prod-analytics-recommend-c1jeg.RED_RECS.GRU_Pytorch_PDP
#"""
  client = bigquery.Client()
  query_job = client.query(query)

# Convert the query result to a pandas DataFrame
  df = query_job.to_dataframe()
  df.to_csv(data_models_path + table_prefix + "_" + table_suffix + '.csv', index=False)

In [ ]:
def load_data(fname, **args):

    with open(fname, 'rt', encoding="utf-8") as f:
        header = f.readline().strip().split('\t')

    if args["session_key"] not in header:
        print(args["session_key"])
        print('ERROR. The colmn specified for session IDs')
        sys.exit(1)
    if args["item_key"] not in header:
        print(args["item_key"])
        print('ERROR. The colmn specified for item IDs')
        sys.exit(1)
    if args["time_key"] not in header:
        print(args["time_key"])
        print('ERROR. The colmn specified for Time')
        sys.exit(1)
    print('Loading data from TAB separated file: {}'.format(fname))

    data = pd.read_csv(fname, sep='\t', usecols=[args["session_key"], args["item_key"],args["time_key"]], dtype={args["session_key"]:'int32', args["item_key"]:'str'})

    return data

In [ ]:
def preprocess_training_data_tmstp(infile, outfile):
    data = pd.read_csv(infile)

    data.columns = ['TimeStr', 'SessionId', 'ItemId']


    print(data.head())
    print(data.dtypes)
    data['TimeStr'] = data['TimeStr'].astype(str)


    item_supports = data.groupby('ItemId').size()
    data = data[np.isin(data.ItemId, item_supports[item_supports>=4].index)]
    del item_supports
    gc.collect()

    data['Time'] = data['TimeStr'].apply(lambda x: datetime.datetime.strptime(x, '%Y-%m-%d_%H_%M').timestamp())

    print("lambda applied")
    del(data['TimeStr'])

    session_lengths = data.groupby('SessionId').size()
    data = data[np.isin(data.SessionId, session_lengths[session_lengths>3].index)]
    del session_lengths
    gc.collect()


    #item_supports = data.groupby('ItemId').size()
    #data = data[np.in1d(data.ItemId, item_supports[item_supports>=2].index)]
    #session_lengths = data.groupby('SessionId').size()
    #data = data[np.in1d(data.SessionId, session_lengths[session_lengths>=2].index)]

    tmax = data.Time.max()
    session_max_times = data.groupby('SessionId').Time.max()
    session_train = session_max_times[session_max_times < tmax-86400].index
    train = data[np.isin(data.SessionId, session_train)]

    del data
    gc.collect()

    print('Full train set\n\tEvents: {}\n\tSessions: {}\n\tItems: {}'.format(len(train), train.SessionId.nunique(), train.ItemId.nunique()))
    train.to_csv(outfile, sep='\t', index=False)

In [ ]:
def predict_gru(mtype, gru, original_train_data, recs_file_name, table_suffix,  device, top_n, data_models_path):
    data = pd.read_csv(original_train_data, sep='\t', usecols=[0,1,2], dtype={0:str, 1:str, 2:str})


    items = data[['ItemId']]
    del data
    gc.collect()
    grouped_items = items.groupby('ItemId')
    del items
    gc.collect()

    distinct_items = grouped_items.count()


    del grouped_items
    gc.collect()


    recs_file = open('/content/out_red_gru4rec.csv', "w")
    recs_file.close()
    recs_file = open('/content/out_red_gru4rec.csv', "a")


    id_map=gru.data_iterator.itemidmap
    id_map_swpapped = pd.Series(id_map.index.values, index=id_map).to_numpy()

    batch_size = 1
    H = []
    for i in range(len(gru.layers)):
        H.append(torch.zeros((batch_size, gru.layers[i]), requires_grad=False, device=gru.device, dtype=torch.float32))


    for row in distinct_items.itertuples():
        i = row.Index
        for h in H: h.detach_()
        in_idx = torch.from_numpy(np.array([id_map[row.Index]])).to(device)
        O = gru.model.forward(in_idx, H, None, training=False)

        oscores = O.T
        O_np = oscores.detach().cpu().numpy()
        O_np = O_np.reshape(-1)
        #top_n = 30
        ind = O_np.argsort()[-31:]
        ind = ind.reshape(-1)
        ind = ind[::-1]
        top_n = id_map_swpapped[ind]

        j = 0
        for t in top_n:
            if str(row.Index) != str(t):
                recs_file.write(str(row.Index) + "," + str(t) + "," + str(O_np[id_map[t]]) + "," + str(j+1) + "\n")
                j=j+1

    recs_file.close()



    recs_file_name = "/content/out_red_gru4rec.csv"
    print('--------made it!----------')
    project_id = 'prod-analytics-recommend-c1jeg'
    dataset_id = 'RED_RECS'
    table_id = 'GRU_Pytorch_PDP_temp'
    file_path = '/content/out_red_gru4rec.csv'

#prod-analytics-recommend-c1jeg.RED_RECS.GRU_Pytorch_PDP
    schema = [
        bigquery.SchemaField("pkey", "INTEGER"),
        bigquery.SchemaField("recs", "INTEGER"),
        bigquery.SchemaField("score", "FLOAT"),
        bigquery.SchemaField("rank", "INTEGER"),
    ]

    #table_id = "prod-analytics-recommend-c1jeg.RED_RECS.gru4rec_ouput_"
    table_id = "prod-analytics-recommend-c1jeg.RED_RECS.output_red_1"

    client = bigquery.Client()

    job_config = bigquery.LoadJobConfig(
        schema=schema,
        skip_leading_rows=1,  # Skip the header row in the CSV file
        source_format=bigquery.SourceFormat.CSV,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )
    table_suffix = ''

    with open(file_path, "rb") as source_file:
        load_job = client.load_table_from_file(source_file, table_id, job_config=job_config)
        load_job.result()

        print(f"Loaded {load_job.output_rows} rows into {table_id}.")

    # add labels, create final tableGA_4_WEB_RECS_DERIVED.RED_LABELS   #GRU_RED_TRAINING_SET expanded-nebula-754.sandbox_crdow.GRU_RED_PRODUCT_LABELS
    table_suffix = ''
    #final_query = "SELECT pkey, product_id AS recs, score, rank FROM (SELECT product_id AS pkey, recs, score, rank FROM `sandbox_crdow.gru4rec_ouput_128`\
    #                JOIN `GA_4_WEB_RECS_DERIVED.RED_LABELS` \
    #                ON pkey=pid) JOIN `GA_4_WEB_RECS_DERIVED.RED_LABELS` ON recs=pid ORDER BY pkey, rank"  prod-analytics-recommend-c1jeg.RED_RECS

    final_query = "SELECT pkey, product_id AS recs, score, rank FROM (SELECT product_id AS pkey, recs, score, rank FROM `prod-analytics-recommend-c1jeg.RED_RECS.output_red_1`\
                    JOIN `prod-analytics-recommend-c1jeg.RED_RECS.GRU_PRODUCT_LABELS` \
                     ON pkey=pid) JOIN `prod-analytics-recommend-c1jeg.RED_RECS.GRU_PRODUCT_LABELS` ON recs=pid ORDER BY pkey, rank"

#prod-analytics-recommend-c1jeg.RED_RECS.GRU_PRODUCT_LABELS
    print(final_query)
    client = bigquery.Client()
    query_config = bigquery.QueryJobConfig()
    query_config.destination = project_id + "." + dataset_id + "." + "GRU_Pytorch_PDP_temp"
    query_config.write_disposition = 'WRITE_TRUNCATE'

    query_job = client.query(final_query, job_config = query_config)
    query_job.result()

In [ ]:
recs_file = open('/content/out_red_gru4rec.csv', "w")
recs_file.close()
t11 = datetime.datetime.now()

base_path = "/content/"

f = open(base_path + 'gru4rec_config.json')
data = json.load(f)
args_model = data["model_args"]
args_data = data["file_args"]
f.close()

#table_suffix = get_training_table_suffix()
data_models_path = args_data["train_data_path"] + table_suffix + "/"

if not os.path.exists(data_models_path):
    os.makedirs(data_models_path)

#gcs_location = "gs://nkhan/gru4rec/" + table_suffix + "/"
#gcs_files = gcs_location + table_prefix + "_" + table_suffix + "_*.csv"

get_bq_training_table()#table_suffix, gcs_files, data_models_path)

original_train_data = data_models_path + table_prefix + "_" + table_suffix + ".csv"
processed_train_data = data_models_path + 'processed_' + table_prefix + "_" + table_suffix + ".csv"
model_file_name = data_models_path + 'gru_' + args_data["mtype"] + '_model_' + table_suffix + ".pt"
recs_file_name = data_models_path + 'gru_' + args_data["mtype"] + '_recs_' + table_suffix + ".csv"

device = args_model["device"]
d = preprocess_training_data_tmstp(original_train_data, processed_train_data)
print("preprocessed")
del d
gc.collect()

gru = GRU4Rec(device=device)
gru.set_params(**args_model)
data = load_data(processed_train_data,**args_data)
del processed_train_data
gc.collect()
print("data loaded")

data.to_csv(data_models_path + "data.csv", sep='\t', index=False)
print("data to csv")



print('-----train------')
t0 = time.time()
gru.fit(data, sample_cache_max_size=sample_store_size, item_key=args_data["item_key"], session_key=args_data["session_key"], time_key=args_data["time_key"])
t1 = time.time()
print('Total training time: {:.2f}s'.format(t1 - t0))
gru.savemodel(model_file_name)
print('-----saved------')

device = args_model["device"]
top_n = 31

print('-----predict------')
predict_gru(args_data["mtype"], gru, data_models_path + 'data.csv', recs_file_name, table_suffix, device, top_n, data_models_path)# gcs_location,
print('-----predict------')

#shutil.rmtree(data_models_path)
#print("Removed " + data_models_path)

t12 = datetime.datetime.now()
print("Total Time = " + str(t12 - t11))


            TimeStr  SessionId  ItemId
0  2026-03-02_21_32     320108  259962
1  2026-05-05_08_30     320284  336106
2  2025-05-26_04_12     320384   97262
3  2025-09-03_02_29     320479  153453
4  2026-02-26_19_37     320510  381335
TimeStr      object
SessionId     int64
ItemId        int64
dtype: object
lambda applied
Full train set
	Events: 34662583
	Sessions: 5936132
	Items: 214561
preprocessed
SET   device                  TO   cuda:0          (type: <class 'str'>)
SET   layers                  TO   [100]           (type: <class 'list'>)
SET   batch_size              TO   32              (type: <class 'int'>)
SET   dropout_p_embed         TO   0.01            (type: <class 'float'>)
SET   dropout_p_hidden        TO   0.005           (type: <class 'float'>)
SET   learning_rate           TO   0.07            (type: <class 'float'>)
SET   momentum                TO   0.5             (type: <class 'float'>)
SET   n_sample                TO   2048            (type: <class 'int'>)
SET 

/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py:460: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.P0 = torch.tensor(pop[self.data_iterator.itemidmap.index.values], dtype=torch.float32, device=self.device)


Epoch1 --> loss: 5.572920 	(3503.40s) 	[256.24 mb/s | 8200 e/s]
Epoch2 --> loss: 4.885514 	(3515.43s) 	[255.36 mb/s | 8172 e/s]
Epoch3 --> loss: 4.753559 	(3518.69s) 	[255.12 mb/s | 8164 e/s]
Epoch4 --> loss: 4.681483 	(3539.11s) 	[253.65 mb/s | 8117 e/s]
Epoch5 --> loss: 4.634583 	(3530.57s) 	[254.27 mb/s | 8137 e/s]
Epoch6 --> loss: 4.599667 	(3508.85s) 	[255.84 mb/s | 8187 e/s]
Epoch7 --> loss: 4.573855 	(3528.23s) 	[254.43 mb/s | 8142 e/s]
Epoch8 --> loss: 4.552174 	(3525.25s) 	[254.65 mb/s | 8149 e/s]
Epoch9 --> loss: 4.534495 	(3533.44s) 	[254.06 mb/s | 8130 e/s]
Epoch10 --> loss: 4.519494 	(3513.00s) 	[255.54 mb/s | 8177 e/s]
Total training time: 35296.09s
-----saved------
-----predict------
--------made it!----------
Loaded 6437575 rows into prod-analytics-recommend-c1jeg.RED_RECS.output_red_1.
SELECT pkey, product_id AS recs, score, rank FROM (SELECT product_id AS pkey, recs, score, rank FROM `prod-analytics-recommend-c1jeg.RED_RECS.output_red_1`                    JOIN `prod-

In [ ]:
"""
import re
import torch
import numpy as np

file_path = '/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py'

with open(file_path, 'r') as file:
    content = file.read()

# Simpler, more aggressive string replacement for the set_() calls
# Handle weight_ih
content = re.sub(
    r"torch\.tensor\(np\.vstack\(m\),\s*device=self\.G\[i\]\.weight_ih\.device\)",
    r"torch.tensor(np.vstack(m), dtype=torch.float32, device=self.G[i].weight_ih.device)",
    content
)
# Handle weight_hh
content = re.sub(
    r"torch\.tensor\(np\.vstack\(m2\),\s*device=self\.G\[i\]\.weight_hh\.device\)",
    r"torch.tensor(np.vstack(m2), dtype=torch.float32, device=self.G[i].weight_hh.device)",
    content
)
# Handle bias_ih
content = re.sub(
    r"torch\.tensor\(np\.hstack\(b\),\s*device=self\.G\[i\]\.bias_ih\.device\)",
    r"torch.tensor(np.hstack(b), dtype=torch.float32, device=self.G[i].bias_ih.device)",
    content
)
# Handle bias_hh
content = re.sub(
    r"torch\.tensor\(np\.hstack\(b2\),\s*device=self\.G\[i\]\.bias_hh\.device\)",
    r"torch.tensor(np.hstack(b2), dtype=torch.float32, device=self.G[i].bias_hh.device)",
    content
)

# Also explicitly ensure _init_numpy_weights returns float32 arrays to be absolutely safe
content = content.replace(
    "def _init_numpy_weights(self, shape):",
    "def _init_numpy_weights(self, shape):\n        return np.zeros(shape, dtype=np.float32) # DUMMY REPLACEMENT IF NEEDED"
)
# A better way to hook into _init_numpy_weights is to modify its return or generation.
content = re.sub(
    r"(sigma \* np\.random\.randn\(\*shape\))",
    r"\1.astype(np.float32)",
    content
)
content = re.sub(
    r"(np\.zeros\(shape\))",
    r"\1.astype(np.float32)",
    content
)

with open(file_path, 'w') as file:
    file.write(content)

print("Aggressive float32 cast applied to numpy arrays and torch.tensor conversions.")
"""

Aggressive float32 cast applied to numpy arrays and torch.tensor conversions.


In [ ]:
"""
import re

file_path = '/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py'
with open(file_path, 'r') as file:
    content = file.read()

# Search for init_parameter_matrix
matches = re.finditer(r'.*init_parameter_matrix.*', content)
found = False
for match in matches:
    print(match.group(0).strip())
    found = True

if not found:
    print('Function init_parameter_matrix not found in the file.')
"""

def init_parameter_matrix(tensor: torch.Tensor, dim0_scale: int = 1, dim1_scale: int = 1):
init_parameter_matrix(self.Wx0.weight, dim1_scale = 3)
init_parameter_matrix(self.Wrz0, dim1_scale = 2)
init_parameter_matrix(self.Wh0, dim1_scale = 1)
init_parameter_matrix(self.E.weight)
init_parameter_matrix(self.G[i].weight_ih, dim1_scale = 3)
init_parameter_matrix(self.G[i].weight_hh, dim1_scale = 3)
init_parameter_matrix(self.Wy.weight)


In [ ]:
"""
import re

file_path = '/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py'
with open(file_path, 'r') as file:
    content = file.read()

# Extract and print the _reset_weights_to_compatibility_mode function
match = re.search(r"def _reset_weights_to_compatibility_mode\(self\):.*?(?=def |\Z)", content, flags=re.DOTALL)
if match:
    print("--- CURRENT STATE OF _reset_weights_to_compatibility_mode ---")
    print(match.group(0))
else:
    print("Function not found.")
"""

--- CURRENT STATE OF _reset_weights_to_compatibility_mode ---
def _reset_weights_to_compatibility_mode(self):
        np.random.seed(42)
        if self.constrained_embedding:
            n_input = self.layers[-1]
        elif self.embedding:
            n_input = self.embedding
            with torch.no_grad():
                self.E.weight.copy_(torch.tensor(self._init_numpy_weights((self.n_items, n_input)), dtype=self.E.weight.dtype, device=self.E.weight.device))
        else:
            n_input = self.n_items
            m = []
            m.append(self._init_numpy_weights((n_input, self.layers[0])))
            m.append(self._init_numpy_weights((n_input, self.layers[0])))
            m.append(self._init_numpy_weights((n_input, self.layers[0])))
            with torch.no_grad():
                self.GE.Wx0.weight.copy_(torch.tensor(np.hstack(m), dtype=self.GE.Wx0.weight.dtype, device=self.GE.Wx0.weight.device))
            m2 = []
            m2.append(self._init_numpy_weights((se

In [ ]:
"""
import re

file_path = '/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py'
with open(file_path, 'r') as file:
    content = file.read()
"""
new_function = """def _reset_weights_to_compatibility_mode(self):
        np.random.seed(42)
        if self.constrained_embedding:
            n_input = self.layers[-1]
        elif self.embedding:
            n_input = self.embedding
            with torch.no_grad():
                self.E.weight.copy_(torch.tensor(self._init_numpy_weights((self.n_items, n_input)), dtype=self.E.weight.dtype, device=self.E.weight.device))
        else:
            n_input = self.n_items
            m = []
            m.append(self._init_numpy_weights((n_input, self.layers[0])))
            m.append(self._init_numpy_weights((n_input, self.layers[0])))
            m.append(self._init_numpy_weights((n_input, self.layers[0])))
            with torch.no_grad():
                self.GE.Wx0.weight.copy_(torch.tensor(np.hstack(m), dtype=self.GE.Wx0.weight.dtype, device=self.GE.Wx0.weight.device))
            m2 = []
            m2.append(self._init_numpy_weights((self.layers[0] , self.layers[0])))
            m2.append(self._init_numpy_weights((self.layers[0] , self.layers[0])))
            with torch.no_grad():
                self.GE.Wrz0.copy_(torch.tensor(np.hstack(m2), dtype=self.GE.Wrz0.dtype, device=self.GE.Wrz0.device))
                self.GE.Wh0.copy_(torch.tensor(self._init_numpy_weights((self.layers[0] , self.layers[0])), dtype=self.GE.Wh0.dtype, device=self.GE.Wh0.device))
                self.GE.Bh0.copy_(torch.zeros((self.layers[0]*3,), dtype=self.GE.Bh0.dtype, device=self.GE.Bh0.device))
        for i in range(self.start, len(self.layers)):
            m = []
            m.append(self._init_numpy_weights((n_input, self.layers[i])))
            m.append(self._init_numpy_weights((n_input, self.layers[i])))
            m.append(self._init_numpy_weights((n_input, self.layers[i])))
            with torch.no_grad():
                self.G[i].weight_ih.copy_(torch.tensor(np.vstack(m), dtype=self.G[i].weight_ih.dtype, device=self.G[i].weight_ih.device))
            m2 = []
            m2.append(self._init_numpy_weights((self.layers[i] , self.layers[i])))
            m2.append(self._init_numpy_weights((self.layers[i] , self.layers[i])))
            m2.append(self._init_numpy_weights((self.layers[i] , self.layers[i])))
            with torch.no_grad():
                self.G[i].weight_hh.copy_(torch.tensor(np.vstack(m2), dtype=self.G[i].weight_hh.dtype, device=self.G[i].weight_hh.device))
                self.G[i].bias_hh.copy_(torch.zeros((self.layers[i]*3,), dtype=self.G[i].bias_hh.dtype, device=self.G[i].bias_hh.device))
                self.G[i].bias_ih.copy_(torch.zeros((self.layers[i]*3,), dtype=self.G[i].bias_ih.dtype, device=self.G[i].bias_ih.device))
        with torch.no_grad():
            self.Wy.weight.copy_(torch.tensor(self._init_numpy_weights((self.n_items, self.layers[-1])), dtype=self.Wy.weight.dtype, device=self.Wy.weight.device))
            self.By.weight.copy_(torch.zeros((self.n_items, 1), dtype=self.By.weight.dtype, device=self.By.weight.device))"""

"""
# Replace the old function with the new one
content = re.sub(r"def _reset_weights_to_compatibility_mode\(self\):.*?(?=def |\Z)", new_function + "\n    ", content, flags=re.DOTALL)

with open(file_path, 'w') as file:
    file.write(content)

print("Successfully replaced _reset_weights_to_compatibility_mode with a type-safe version.")
"""

Successfully replaced _reset_weights_to_compatibility_mode with a type-safe version.


In [ ]:
"""
import re

file_path = '/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py'
with open(file_path, 'r') as file:
    content = file.read()

# Extract and print the _reset_weights_to_compatibility_mode function
match = re.search(r"def _reset_weights_to_compatibility_mode\(self\):.*?(?=def |\Z)", content, re.DOTALL)
if match:
    print("--- CURRENT STATE OF _reset_weights_to_compatibility_mode ---")
    print(match.group(0))
else:
    print("Function not found.")
"""

--- CURRENT STATE OF _reset_weights_to_compatibility_mode ---
def _reset_weights_to_compatibility_mode(self):
        np.random.seed(42)
        if self.constrained_embedding:
            n_input = self.layers[-1]
        elif self.embedding:
            n_input = self.embedding
            self.E.weight.set_(torch.tensor(self._init_numpy_weights((self.n_items, n_input)), device=self.E.weight.device))
        else:
            n_input = self.n_items
            m = []
            m.append(self._init_numpy_weights((n_input, self.layers[0])))
            m.append(self._init_numpy_weights((n_input, self.layers[0])))
            m.append(self._init_numpy_weights((n_input, self.layers[0])))
            self.GE.Wx0.weight.set_(torch.tensor(np.hstack(m), device=self.GE.Wx0.weight.device))
            m2 = []
            m2.append(self._init_numpy_weights((self.layers[0] , self.layers[0])))
            m2.append(self._init_numpy_weights((self.layers[0] , self.layers[0])))
            self.GE.W

In [ ]:
"""
import re

file_path = '/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py'

with open(file_path, 'r') as file:
    content = file.read()

# Let's find the mangled _init_numpy_weights and replace it cleanly.
# We will use a regular expression to match the definition and its body.

# The pattern looks for the def and everything up to the next method definition (def )
pattern = r"def _init_numpy_weights\(self, shape\):.*?return m"

# If my previous mangling ruined the structure, let's just do a more targeted replace.
# Let's replace the dummy return we injected earlier:
content = content.replace(
    "return np.zeros(shape, dtype=np.float32) # DUMMY REPLACEMENT IF NEEDED\n",
    ""
)

# Ensure the random initialization casts to float32 at the end.
# It originally looked something like:
# m = np.random.rand(*shape) * 2 * sigma - sigma
# return m
"""
# Let's just forcefully replace the whole function using regex to be safe
clean_function = """def _init_numpy_weights(self, shape):
        sigma = np.sqrt(6.0 / (shape[0] + shape[1]))
        m = np.random.rand(*shape) * 2.0 * sigma - sigma
        return m.astype(np.float32)"""

"""
content = re.sub(r"def _init_numpy_weights\(self, shape\):\s*(?:return np\.zeros.*?\n)?\s*sigma = .*?\n.*?return m", clean_function, content, flags=re.DOTALL)

with open(file_path, 'w') as file:
    file.write(content)

print("Cleaned up _init_numpy_weights to properly calculate random weights and return as float32.")
"""

Cleaned up _init_numpy_weights to properly calculate random weights and return as float32.


In [ ]:
"""
import re

file_path = '/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py'

with open(file_path, 'r') as file:
    content = file.read()

# Replace strict .set_(...) with the safer .data.copy_(...)
content = re.sub(
    r"self\.G\[i\]\.(weight_ih|weight_hh|bias_ih|bias_hh)\.set_\((.*?)\)",
    r"self.G[i].\1.data.copy_(\2)",
    content
)

with open(file_path, 'w') as file:
    file.write(content)

print("Replaced strict .set_() calls with .data.copy_() in gru4rec_pytorch.py")
"""

<>:11: SyntaxWarning: invalid escape sequence '\.'
<>:11: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_155/1557862285.py:11: SyntaxWarning: invalid escape sequence '\.'
  r"self\.G\[i\]\.(weight_ih|weight_hh|bias_ih|bias_hh)\.set_\((.*?)\)",


'\nimport re\n\nfile_path = \'/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py\'\n\nwith open(file_path, \'r\') as file:\n    content = file.read()\n\n# Replace strict .set_(...) with the safer .data.copy_(...)\ncontent = re.sub(\n    r"self\\.G\\[i\\]\\.(weight_ih|weight_hh|bias_ih|bias_hh)\\.set_\\((.*?)\\)",\n    r"self.G[i].\x01.data.copy_(\x02)",\n    content\n)\n\nwith open(file_path, \'w\') as file:\n    file.write(content)\n\nprint("Replaced strict .set_() calls with .data.copy_() in gru4rec_pytorch.py")\n'

In [ ]:
device = args_model["device"]
top_n = 31

print('-----predict------')
predict_gru(args_data["mtype"], gru, data_models_path + 'data.csv', recs_file_name, table_suffix, device, top_n, data_models_path)# gcs_location,
print('-----predict------')

#shutil.rmtree(data_models_path)
#print("Removed " + data_models_path)

t12 = datetime.datetime.now()
print("Total Time = " + str(t12 - t11))

-----predict------
--------made it!----------
--------made it!----------
Loaded 10591320 rows into prod-analytics-recommend-c1jeg.RED_RECS.output_red_.
SELECT pkey, product_id AS recs, score, rank FROM (SELECT product_id AS pkey, recs, score, rank FROM `prod-analytics-recommend-c1jeg.RED_RECS.output_red_`                    JOIN `prod-analytics-recommend-c1jeg.RED_RECS.GRU_PRODUCT_LABELS`                      ON pkey=pid) JOIN `prod-analytics-recommend-c1jeg.RED_RECS.GRU_PRODUCT_LABELS` ON recs=pid ORDER BY pkey, rank
Loaded 10591320 rows into prod-analytics-recommend-c1jeg.RED_RECS.output_red_.
SELECT pkey, product_id AS recs, score, rank FROM (SELECT product_id AS pkey, recs, score, rank FROM `prod-analytics-recommend-c1jeg.RED_RECS.output_red_`                    JOIN `prod-analytics-recommend-c1jeg.RED_RECS.GRU_PRODUCT_LABELS`                      ON pkey=pid) JOIN `prod-analytics-recommend-c1jeg.RED_RECS.GRU_PRODUCT_LABELS` ON recs=pid ORDER BY pkey, rank
-----predict------
Total 

In [ ]:
def preprocess_training_data(infile, outfile):

    data = pd.read_csv(infile, sep=',', header=0, usecols=[0,1,2], dtype={0:np.int32, 1:np.int64, 2:np.int64})
    data.columns = ['SessionId', 'Time', 'ItemId']
    #data['Time'] = data.TimeStr.apply(lambda x: datetime.datetime.strptime(x, '%Y%m%d_%H_%M').timestamp())
    #del(data['TimeStr'])

    session_lengths = data.groupby('SessionId').size()
    data = data[np.isin(data.SessionId, session_lengths[session_lengths>1].index)]
    item_supports = data.groupby('ItemId').size()
    data = data[np.isin(data.ItemId, item_supports[item_supports>=1].index)]
    session_lengths = data.groupby('SessionId').size()
    data = data[np.isin(data.SessionId, session_lengths[session_lengths>=1].index)]

    tmax = data.Time.max()
    session_max_times = data.groupby('SessionId').Time.max()
    session_train = session_max_times[session_max_times < tmax-15].index
    train = data[np.isin(data.SessionId, session_train)]

    print('Full train set\n\tEvents: {}\n\tSessions: {}\n\tItems: {}'.format(len(train), train.SessionId.nunique(), train.ItemId.nunique()))
    train.to_csv(outfile, sep='\t', index=False)
    return train

In [ ]:
def get_training_table_suffix():

    query = ('SELECT MAX(TABLE_SUFFIX) AS ts FROM `' + project + '.' + dataset + '.' + table_prefix + '`')

    client = bigquery.Client()
    query_job = client.query(query)
    rows = query_job.result()

    table_suffix = ""
    for row in rows:
        table_suffix = row[0]
        print(table_suffix)

    return table_suffix